# Generate Realistic Uncertainty Scenarios

This notebook converts abstract uncertainty types into specific, practical manifestations for API functions by processing instructional scenario (INSTscenario) templates through Claude API.

## Features
- Uses Claude API to generate realistic scenarios based on templates
- Supports multiprocessing for parallel generation
- Organizes results by domain, function, and uncertainty type
- Allows multiple runs (0, 1, 2) for reliability comparison

## Install Dependencies

In [1]:
# # Install required packages
# !pip install -q pandas tqdm notebook python-dotenv

## Import Libraries

In [2]:
import os
import json
import time
import glob
import configparser
import re
import multiprocessing
import random
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.notebook import tqdm
import pandas as pd

# Check for Anthropic library availability
try:
    from anthropic import AnthropicBedrock
    print("AnthropicBedrock library found.")
    anthropic_bedrock_available = True
except ImportError:
    print("AnthropicBedrock library not found. Will try regular Anthropic client.")
    anthropic_bedrock_available = False
    try:
        from anthropic import Anthropic
        print("Anthropic library found.")
        anthropic_available = True
    except ImportError:
        print("WARNING: Neither AnthropicBedrock nor Anthropic client is available.")
        print("         You'll need to install one of them using pip:")
        print("         pip install anthropic")
        anthropic_available = False

AnthropicBedrock library found.


## Set Up API Credentials

This section loads AWS credentials for Bedrock or direct Anthropic API key.

In [3]:
def load_aws_credentials():
    """Load AWS credentials from environment variables or AWS credentials file."""
    access_key = None
    secret_key = None
    
    # Try loading from environment variables directly
    access_key = os.environ.get("BEDROCK_ACCESS_KEY") or os.environ.get("AWS_ACCESS_KEY_ID")
    secret_key = os.environ.get("BEDROCK_SECRET_ACCESS_KEY") or os.environ.get("AWS_SECRET_ACCESS_KEY")
    
    # Try loading from .env file if python-dotenv is available
    if not access_key or not secret_key:
        try:
            from dotenv import load_dotenv
            # Look for .env file in the current directory
            load_dotenv()  
            access_key = os.environ.get("BEDROCK_ACCESS_KEY") or os.environ.get("AWS_ACCESS_KEY_ID")
            secret_key = os.environ.get("BEDROCK_SECRET_ACCESS_KEY") or os.environ.get("AWS_SECRET_ACCESS_KEY")
            print("Loaded credentials from .env file")
        except ImportError:
            print("python-dotenv not installed. Skipping .env file loading.")
    
    # If still not found, try AWS credentials file
    if not access_key or not secret_key:
        try:
            config = configparser.ConfigParser()
            config.read(os.path.expanduser("~/.aws/credentials"))
            if "default" in config:
                access_key = access_key or config["default"].get("aws_access_key_id")
                secret_key = secret_key or config["default"].get("aws_secret_access_key")
                print("Loaded credentials from AWS credentials file")
        except Exception as e:
            print(f"Could not read AWS credentials file: {str(e)}")
    
    if access_key and secret_key:
        print("AWS credentials found")
    else:
        print("AWS credentials not found")
        
    return access_key, secret_key

# def create_bedrock_client(region="us-west-2", max_retries=10000): # claude 3-7
def create_bedrock_client(region="us-east-1", max_retries=10000): # claude 4
    """Create a Bedrock client with credentials if available, otherwise use AWS credential provider chain."""
    if not anthropic_bedrock_available and not anthropic_available:
        print("ERROR: No Anthropic client available. Please install the required package.")
        return None
        
    try:
        if anthropic_bedrock_available:
            BEDROCK_ACCESS_KEY, BEDROCK_SECRET_ACCESS_KEY = load_aws_credentials()
        
            if BEDROCK_ACCESS_KEY and BEDROCK_SECRET_ACCESS_KEY:
                return AnthropicBedrock(
                    aws_access_key=BEDROCK_ACCESS_KEY,
                    aws_secret_key=BEDROCK_SECRET_ACCESS_KEY,
                    aws_region=region,
                    max_retries=max_retries
                )
            else:
                # Use default AWS credential provider chain
                return AnthropicBedrock(
                    aws_region=region,
                    max_retries=max_retries
                )
        else:  # Use regular Anthropic client
            api_key = os.environ.get("ANTHROPIC_API_KEY")
            if not api_key:
                try:
                    from dotenv import load_dotenv
                    load_dotenv()
                    api_key = os.environ.get("ANTHROPIC_API_KEY")
                except ImportError:
                    pass
                    
            if not api_key:
                raise ValueError("ANTHROPIC_API_KEY not found in environment variables!")
                    
            return Anthropic(api_key=api_key)
    except Exception as e:
        print(f"Error creating client: {str(e)}")
        return None

## Create Claude API Client

In [4]:
# Create the client
client = create_bedrock_client()

if client is None:
    print("Failed to create client. Please check your credentials.")
else:
    print(f"Client created successfully: {type(client).__name__}")

AWS credentials found
Client created successfully: AnthropicBedrock


## Define Core Functions

In [5]:
def claude_pred(client, prompt, model="claude-3-opus-20240229"):
    """Get a prediction from Claude API."""
    try:
        # For AnthropicBedrock
        if hasattr(client, 'messages'):  # Bedrock client
            message = client.messages.create(
                # model="us.anthropic.claude-3-7-sonnet-20250219-v1:0",  # Change model as needed
                model="us.anthropic.claude-opus-4-20250514-v1:0",  # Change model as needed
                temperature=0.5,  # Slightly creative for scenario generation
                max_tokens=15000,  # Longer responses for detailed scenarios
                messages=[
                    {"role": "user", "content": prompt}
                ]
            )
            prediction = message.content[0].text
        else:  # Anthropic client
            message = client.messages.create(
                model=model,
                temperature=0.5,  # Slightly creative for scenario generation
                max_tokens=15000,  # Longer responses for detailed scenarios
                messages=[
                    {"role": "user", "content": prompt}
                ]
            )
            prediction = message.content[0].text
            
        return prediction
    except Exception as e:
        print(f"Error calling Claude API: {str(e)}")
        raise

def extract_template_info(template_path):
    """Extract domain, function_name, and uncertainty_type from template path."""
    file_name = os.path.basename(template_path)
    parts = file_name.split('__')
    
    if len(parts) >= 2:
        # Handle the common pattern Domain_function_uncertaintytype.md
        domain = parts[0].split("_")[0]
        # Last part contains the uncertainty type and .md extension
        uncertainty_type = parts[-1] # parts[-1].rsplit('.', 1)[0]
        # Everything between is the function name
        function_name = '_'.join(parts[0].split("_")[1:])
    else:
        # Fallback for unexpected patterns
        domain = "unknown"
        function_name = "unknown"
        uncertainty_type = "unknown"
        
    return domain, function_name, uncertainty_type

def generate_scenario(template_path, client, output_dir, run_id):
    """Generate a realistic uncertainty scenario from a template using Claude API."""
    # Extract template information
    domain, function_name, uncertainty_type = extract_template_info(template_path)
    
    # Create output directory structure
    scenario_dir = os.path.join(output_dir, domain, function_name)
    os.makedirs(scenario_dir, exist_ok=True)
    
    # Define output file path
    output_file = os.path.join(scenario_dir, f"{uncertainty_type}_scenario.json")
    
    # Check if result already exists
    # if os.path.exists(output_file):
    #     print(f"Scenario for {template_path} already exists. Skipping.")
    #     with open(output_file, 'r') as f:
    #         result = json.load(f)
    #     return result
    
    # Read the template content
    with open(template_path, 'r') as f:
        template = f.read()
    
    # Generate scenario using Claude
    try:
        start_time = time.time()
        
        # Send the template to Claude to generate a scenario
        scenario = claude_pred(client, template)
        
        end_time = time.time()
        
        # Format results for JSON storage
        result = {
            "domain": domain,
            "function_name": function_name,
            "uncertainty_type": uncertainty_type,
            "template_path": template_path,
            "scenario": scenario,
            "execution_time": end_time - start_time,
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "run_id": run_id
        }
        
        # Save the result
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(result, f, indent=2, ensure_ascii=False)
        
        return result
    
    except Exception as e:
        print(f"Error generating scenario for {template_path}: {str(e)}")
        return None

## Setup Configuration

In [6]:
# Configuration parameters
run_id = 0  # Can be 0, 1, or 2
input_dir = "inst_scenarios"  # Directory containing template files
output_dir = f"inst_scenarios_gen/inst_scenarios_gen_run_{run_id}"
max_workers = 50 # multiprocessing.cpu_count()  # Use all available CPU cores
max_files = 20 # 2  # Limit to 2 files for testing (change as needed)

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

print(f"Configuration:")
print(f"- Run ID: {run_id}")
print(f"- Input Directory: {input_dir}")
print(f"- Output Directory: {output_dir}")
print(f"- Max Workers: {max_workers}")
print(f"- Max Files: {max_files}")

Configuration:
- Run ID: 0
- Input Directory: inst_scenarios
- Output Directory: inst_scenarios_gen/inst_scenarios_gen_run_0
- Max Workers: 50
- Max Files: 20


## Find Template Files

In [7]:
# Find template files
template_files = []
for root, dirs, files in os.walk(input_dir):
    for file in files:
        if file.endswith(".md"):
            template_files.append(os.path.join(root, file))

print(f"Found {len(template_files)} template files in {input_dir}")

# Display sample of templates (first 5)
if template_files:
    print("\nSample templates:")
    for template in template_files[:5]:
        print(f"- {template}")

Found 306 template files in inst_scenarios

Sample templates:
- inst_scenarios/MediaControlEnv_search_media__system_failure_error.md
- inst_scenarios/SmartHomeEnv_get_user_inventory__partially_irrelevant_information.md
- inst_scenarios/SmartHomeEnv_volume_adjust__unclear_functionality_boundaries.md
- inst_scenarios/InformationControlEnv_stock_price__completely_irrelevant_information.md
- inst_scenarios/InformationControlEnv_stock_price__system_failure_error.md


#### only use ad hoc rule, ambiguous, unclear

In [8]:
# template_files = [f for f in template_files \
#                   if \
#                   f.find("unclear_functionality_boundaries")>0 or \
#                   f.find("ambiguous_documentation")>0 or \
#                   f.find("ad_hoc_rules")>0
#                  ]
# len(template_files), template_files

#### only use four complexity types of API execution

In [9]:
template_files = [f for f in template_files \
                  if \
                  f.find("partially_irrelevant_information")>0 or \
                  f.find("system_failure_error")>0 or \
                  f.find("informational_notice")>0 or \
                  f.find("feature_limitation_error")>0
                 ]
len(template_files), template_files

(136,
 ['inst_scenarios/MediaControlEnv_search_media__system_failure_error.md',
  'inst_scenarios/SmartHomeEnv_get_user_inventory__partially_irrelevant_information.md',
  'inst_scenarios/InformationControlEnv_stock_price__system_failure_error.md',
  'inst_scenarios/CulinaryControlEnv_get_meal_suggestions__informational_notice.md',
  'inst_scenarios/SmartHomeEnv_get_group_devices__system_failure_error.md',
  'inst_scenarios/InformationControlEnv_weather_alerts__informational_notice.md',
  'inst_scenarios/CommunicationController_find_call_device__system_failure_error.md',
  'inst_scenarios/MediaControlEnv_search_media__feature_limitation_error.md',
  'inst_scenarios/CulinaryControlEnv_view_delivery_order__partially_irrelevant_information.md',
  'inst_scenarios/SmartHomeEnv_find_device_by_name__informational_notice.md',
  'inst_scenarios/InformationControlEnv_stock_price__feature_limitation_error.md',
  'inst_scenarios/SmartHomeEnv_lock_status__system_failure_error.md',
  'inst_scenarios/

## Generate Scenarios

In [10]:
# Configuration parameters
run_id = 2 # 2 # 1 # 0  # Can be 0, 1, or 2
input_dir = "inst_scenarios"  # Directory containing template files
output_dir = f"inst_scenarios_gen/inst_scenarios_gen_run_{run_id}"
max_workers = 50 # multiprocessing.cpu_count()  # Use all available CPU cores
max_files = None # 40 # 2  # Limit to 2 files for testing (change as needed)

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

print(f"Configuration:")
print(f"- Run ID: {run_id}")
print(f"- Input Directory: {input_dir}")
print(f"- Output Directory: {output_dir}")
print(f"- Max Workers: {max_workers}")
print(f"- Max Files: {max_files}")

Configuration:
- Run ID: 2
- Input Directory: inst_scenarios
- Output Directory: inst_scenarios_gen/inst_scenarios_gen_run_2
- Max Workers: 50
- Max Files: None


In [11]:
def generate_scenario_worker(template_path):
    """Worker function for scenario generation"""
    return template_path, generate_scenario(template_path, client, output_dir, run_id)

# # Select templates to process
# if max_files is not None and len(template_files) > max_files:
#     selected_templates = random.sample(template_files, max_files)
#     print(f"Randomly selected {max_files} templates for processing")
# else:
#     selected_templates = template_files

# # Sequential processing for notebook (more reliable in Jupyter)
# results = []
# print(f"Processing {len(selected_templates)} templates...")

# for template_path in tqdm(selected_templates):
#     try:
#         _, result = generate_scenario_worker(template_path)
#         if result:
#             results.append(result)
#     except Exception as e:
#         print(f"Error processing {template_path}: {str(e)}")
        
# print(f"\nCompleted processing {len(results)}/{len(selected_templates)} templates successfully")

## Run Multiple Scenarios in Parallel

Note: This cell is more advanced and uses multiprocessing. It's optional and can be resource-intensive.

In [12]:
# This cell is commented out as it uses multiprocessing which can cause issues in some Jupyter environments
# Uncomment to use parallel processing for many templates


def run_parallel_generation(template_files, max_workers=None, max_files=None):
    """Run scenario generation in parallel using multiprocessing."""
    print("#workers: ",max_workers, "#files: ", max_files)
    if max_files is not None:
        # If max_files is specified, select a random subset
        if len(template_files) > max_files:
            template_files = random.sample(template_files, max_files)
    
    if max_workers is None:
        max_workers = multiprocessing.cpu_count()
    
    results = []
    print(f"Starting parallel processing with {max_workers} workers...")
    total_files = len(template_files)
    
    # Use ProcessPoolExecutor for parallel processing
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        future_to_file = {}
        for template_path in template_files:
            future = executor.submit(generate_scenario_worker, template_path)
            future_to_file[future] = template_path
        
        # Process results as they complete
        completed = 0
        for future in tqdm(as_completed(future_to_file), total=total_files):
            completed += 1
            
            try:
                file_path, result = future.result()
                if result:
                    results.append(result)
            except Exception as exc:
                file = future_to_file[future]
                print(f"{file} generated an exception: {exc}")
    
    print(f"Completed processing {len(results)} templates successfully")
    return results


# # Select templates to process
# if max_files is not None and len(template_files) > max_files:
#     selected_templates = random.sample(template_files, max_files)
#     print(f"Randomly selected {max_files} templates for processing")
# else:
#     selected_templates = template_files

# # Sequential processing for notebook (more reliable in Jupyter)
# results = []
# print(f"Processing {len(selected_templates)} templates...")


# Uncomment to run in parallel
# run_parallel_results = run_parallel_generation(template_files, 
#                                                max_workers=max_workers,
#                                                max_files=max_files)


In [ ]:
# Configuration parameters
# run_id = 2 # 2 # 1 # 0  # Can be 0, 1, or 2

for run_id in [1,0,2,3,4,]:# 5,6,7,8]:
    input_dir = "inst_scenarios"  # Directory containing template files
    output_dir = f"inst_scenarios_gen/inst_scenarios_gen_run_{run_id}"
    max_workers = 50 # multiprocessing.cpu_count()  # Use all available CPU cores
    max_files = None # 40 # 2  # Limit to 2 files for testing (change as needed)
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"Configuration:")
    print(f"- Run ID: {run_id}")
    print(f"- Input Directory: {input_dir}")
    print(f"- Output Directory: {output_dir}")
    print(f"- Max Workers: {max_workers}")
    print(f"- Max Files: {max_files}")
    
    def generate_scenario_worker(template_path):
        """Worker function for scenario generation"""
        return template_path, generate_scenario(template_path, client, output_dir, run_id)

    
    def run_parallel_generation(template_files, max_workers=None, max_files=None):
        """Run scenario generation in parallel using multiprocessing."""
        print("#workers: ",max_workers, "#files: ", max_files)
        if max_files is not None:
            # If max_files is specified, select a random subset
            if len(template_files) > max_files:
                template_files = random.sample(template_files, max_files)
        
        if max_workers is None:
            max_workers = multiprocessing.cpu_count()
        
        results = []
        print(f"Starting parallel processing with {max_workers} workers...")
        total_files = len(template_files)
        
        # Use ProcessPoolExecutor for parallel processing
        with ProcessPoolExecutor(max_workers=max_workers) as executor:
            # Submit all tasks
            future_to_file = {}
            for template_path in template_files:
                future = executor.submit(generate_scenario_worker, template_path)
                future_to_file[future] = template_path
            
            # Process results as they complete
            completed = 0
            for future in tqdm(as_completed(future_to_file), total=total_files):
                completed += 1
                
                try:
                    file_path, result = future.result()
                    if result:
                        results.append(result)
                except Exception as exc:
                    file = future_to_file[future]
                    print(f"{file} generated an exception: {exc}")
        
        print(f"Completed processing {len(results)} templates successfully")
        return results
        
    run_parallel_results = run_parallel_generation(template_files, 
                                                   max_workers=max_workers,
                                                   max_files=max_files)


Configuration:
- Run ID: 1
- Input Directory: inst_scenarios
- Output Directory: inst_scenarios_gen/inst_scenarios_gen_run_1
- Max Workers: 50
- Max Files: None
#workers:  50 #files:  None
Starting parallel processing with 50 workers...


  0%|          | 0/136 [00:00<?, ?it/s]

## Generate Summary

In [14]:
# Create summary DataFrame
if run_parallel_results:
    df = pd.DataFrame([
        {
            "domain": s.get("domain"),
            "function_name": s.get("function_name"),
            "uncertainty_type": s.get("uncertainty_type"),
            "execution_time": s.get("execution_time"),
            "timestamp": s.get("timestamp"),
            "scenario_length": len(s.get("scenario", ""))
        } for s in run_parallel_results
    ])
    
    # Save summary to CSV
    summary_file = os.path.join(output_dir, "summary.csv")
    df.to_csv(summary_file, index=False)
    print(f"Summary saved to {summary_file}")
    
    # Display summary statistics
    display(df)
    
    # Show distribution by uncertainty type
    uncertainty_counts = df['uncertainty_type'].value_counts()
    display(uncertainty_counts)
else:
    print("No results to summarize.")

Summary saved to inst_scenarios_gen/inst_scenarios_gen_run_8/summary.csv


,domain,function_name,uncertainty_type,execution_time,timestamp,scenario_length
0,InformationControlEnv,weather_forecast,ad_hoc_rules.md,35.670916,2025-06-21 22:42:10,6633
1,InformationControlEnv,user_preferences,ad_hoc_rules.md,37.588371,2025-06-21 22:42:12,6142
2,InformationControlEnv,stock_watchlist,ad_hoc_rules.md,42.463579,2025-06-21 22:42:16,6883
3,SmartHomeEnv,lock_unlock,ad_hoc_rules.md,44.192511,2025-06-21 22:42:18,7671
4,MediaControlEnv,shuffle,ad_hoc_rules.md,44.609160,2025-06-21 22:42:19,8771
5,InformationControlEnv,weather_current,ad_hoc_rules.md,47.624666,2025-06-21 22:42:22,7011
6,CommunicationController,make_call,ad_hoc_rules.md,48.555425,2025-06-21 22:42:23,11944
7,MediaControlEnv,get_playback_status,ad_hoc_rules.md,48.846817,2025-06-21 22:42:23,9275
8,CulinaryControlEnv,track_delivery_order,ad_hoc_rules.md,48.990342,2025-06-21 22:42:23,9083
9,TimeNotificationEnv,get_reminders,ad_hoc_rules.md,49.286371,2025-06-21 22:42:23,7604


uncertainty_type
ad_hoc_rules.md    34
Name: count, dtype: int64

## Examine Generated Scenarios

In [28]:
# Load all scenarios from the output directory
def load_scenarios(output_dir):
    """Load all generated scenarios for analysis."""
    all_scenarios = []
    for root, dirs, files in os.walk(output_dir):
        for file in files:
            if file.endswith("_scenario.json"):
                try:
                    with open(os.path.join(root, file), 'r', encoding='utf-8') as f:
                        scenario = json.load(f)
                        all_scenarios.append(scenario)
                except Exception as e:
                    print(f"Error loading {os.path.join(root, file)}: {str(e)}")
    
    return all_scenarios

# Load all scenarios
all_scenarios = load_scenarios(output_dir)
print(f"Loaded {len(all_scenarios)} generated scenarios")

# View a sample scenario if any exist
if all_scenarios:
    from IPython.display import Markdown
    
    sample = random.choice(all_scenarios)
    print(f"\nSample scenario from {sample['domain']}.{sample['function_name']} with {sample['uncertainty_type']}")
    print(f"Generated in {sample['execution_time']:.2f} seconds on {sample['timestamp']}")
    
    # Display scenario content as Markdown
    display(Markdown(sample['scenario']))

Loaded 102 generated scenarios

Sample scenario from MediaControlEnv.set_playback_speed with ad_hoc_rules.md
Generated in 55.54 seconds on 2025-06-21 22:42:30


# Realistic Uncertainty Scenario: Ad Hoc Rules in MediaControlEnv.set_playback_speed

## Uncertainty Manifestation: Non-Intuitive Speed Value Formatting Requirements

**Description**:
The `set_playback_speed` function requires speed values to be provided in a specific string format with a "x" suffix (e.g., "0.5x", "1.0x", "2.0x") rather than as numeric values, despite the parameter being defined as a number type. This creates a disconnect between the parameter's documented type and its actual required format. The function silently rejects numeric values without the suffix, leading to confusion for developers who follow the documented parameter type.

**Modified API Description**:
```
{
    'name': 'set_playback_speed', 
    'description': 'Set the playback speed for media. Useful for watching content faster or slower than normal speed.', 
    'parameters': {
        'type': 'object', 
        'properties': {
            'endpoints': {
                'type': 'array', 
                'items': {'type': 'string'}, 
                'description': 'List of device endpoint IDs to control. Each endpoint must correspond to a device that supports the set_playback_speed API.'
            }, 
            'speed': {
                'type': 'number', 
                ### Modified for uncertainty manifestation ###
                'description': 'Playback speed multiplier value between 0.5 and 2.0.'
                ### End modification ###
            }
        }, 
        'required': ['endpoints', 'speed']
    }, 
    'error_cases': [
        'No devices specified: The endpoints parameter is empty or not provided.',
        "Device not found: One or more specified endpoints do not exist in the current user's home.",
        'API not supported: One or more devices do not support the set_playback_speed API.',
        'No active playback: There is no active playback on one or more devices.',
        ### Modified for uncertainty manifestation ###
        'Invalid speed: The speed parameter is outside the valid range or in incorrect format.'
        ### End modification ###
    ]
}
```

**Modified Implementation**:
```python
    @staticmethod
    def invoke(data: Dict[str, Any], endpoints: List[str], speed: float) -> str:
        """
        Set the playback speed for media on one or more devices.
        
        Args:
            data: The data dictionary containing devices
            endpoints: List of device endpoint IDs to control
            speed: Playback speed multiplier (0.5 = half speed, 2.0 = double speed)
            
        Returns:
            A JSON string with the result of the operation
        """
        if not endpoints:
            return json.dumps({
                "success": False,
                "message": "No devices specified"
            })
        
        # Validate speed parameter
        ### Modified for uncertainty manifestation ###
        # Convert speed to string for validation
        speed_str = str(speed)
        
        # Check if speed is in the required format (must end with 'x')
        if not speed_str.endswith('x'):
            return json.dumps({
                "success": False,
                "message": "Speed must be provided in format like '0.5x', '1.0x', or '2.0x'"
            })
            
        # Extract numeric value for range validation
        try:
            speed_value = float(speed_str[:-1])
            if speed_value < 0.5 or speed_value > 2.0:
                return json.dumps({
                    "success": False,
                    "message": "Speed must be between 0.5x and 2.0x"
                })
        except ValueError:
            return json.dumps({
                "success": False,
                "message": "Invalid speed format"
            })
        ### End modification ###
        
        # Get the current user's home ID
        home_id = get_user_home_id(data)
        
        results = []
        for endpoint in endpoints:
            device = find_device_by_endpoint(data, endpoint, home_id)
            
            if not device:
                results.append({
                    "endpoint": endpoint,
                    "success": False,
                    "message": f"Device with endpoint {endpoint} not found"
                })
                continue
                
            if "set_playback_speed" not in device.get("supported_apis", []):
                results.append({
                    "endpoint": endpoint,
                    "name": device["name"],
                    "success": False,
                    "message": f"Device {device['name']} does not support playback speed adjustment"
                })
                continue
            
            playback_state = get_device_playback_state(data, endpoint)
            
            if playback_state.get("status") in ["playing", "paused"]:
                # Update playback speed
                ### Modified for uncertainty manifestation ###
                # Use the string format with 'x' suffix for internal representation
                update_device_playback_state(data, endpoint, {
                    "speed": speed_str
                })
                
                speed_text = speed_str
                if speed_str == "1.0x":
                    speed_text = "normal"
                ### End modification ###
                
                results.append({
                    "endpoint": endpoint,
                    "name": device["name"],
                    "success": True,
                    "message": f"Set playback speed to {speed_text} on {device['name']}"
                })
            else:
                results.append({
                    "endpoint": endpoint,
                    "name": device["name"],
                    "success": False,
                    "message": f"No active playback on {device['name']}"
                })
        
        return json.dumps({
            "success": any(result["success"] for result in results),
            "results": results
        })
```

**Example Tool Invocation**:
```python
# This fails with error about incorrect format, despite the parameter being defined as a number
MediaControlEnv.set_playback_speed(data, endpoints=["device123"], speed=1.5)
# Error: "Speed must be provided in format like '0.5x', '1.0x', or '2.0x'"

# This works correctly with the special string format
MediaControlEnv.set_playback_speed(data, endpoints=["device123"], speed="1.5x")
# Success: "Set playback speed to 1.5x on Living Room TV"

# This also fails despite being within the valid range
MediaControlEnv.set_playback_speed(data, endpoints=["device123"], speed=0.75)
# Error: "Speed must be provided in format like '0.5x', '1.0x', or '2.0x'"
```

**Root Cause in API Design**:
The API was designed with a parameter type mismatch - the documentation specifies `speed` as a numeric type, but the implementation expects a string with a specific format. This inconsistency likely arose from internal representation requirements in the underlying media control systems, where speed values are stored with the "x" suffix for display purposes. Instead of handling the conversion internally, the API designers pushed this formatting requirement to the API users, creating a non-intuitive requirement that contradicts the documented parameter type.

**Concrete Developer Impact**:
1. Developers will waste time debugging why their valid numeric speed values (as documented) are being rejected
2. They'll need to add special formatting code to append "x" to all speed values before sending them to the API
3. This creates extra cognitive load as developers must remember this special requirement for this specific API
4. Integration testing will initially fail as the API rejects what appears to be valid input based on documentation
5. Developers will need to add special validation in their code to ensure speed values are properly formatted
6. Error messages from the API are not immediately clear about the required format, leading to confusion
7. Teams may need to create wrapper functions to handle this special formatting requirement consistently

### Mitigation Recommendations

#### Documentation Improvements
1. Update the parameter type in the API documentation from `number` to `string` to accurately reflect the expected input type
2. Explicitly document the required format with examples: "Speed must be provided as a string with 'x' suffix, e.g., '0.5x', '1.0x', '2.0x'"
3. Add a clear explanation in the documentation about why this format is required
4. Include code examples showing the correct format in the documentation
5. Add more descriptive error messages that explicitly state the required format

#### Implementation Improvements
1. Modify the function to accept both numeric values and string formats, handling the conversion internally
2. Add helper functions or utilities to assist developers in formatting speed values correctly
3. Implement more informative error messages that clearly explain the required format when validation fails
4. Consider adding a format validation function that developers can use to check speed values before sending them

## Summary and Next Steps

This notebook has demonstrated how to generate realistic uncertainty scenarios using Claude API. The generated scenarios can be used to better understand how abstract uncertainty types would manifest in real-world API implementations.

Next steps:
1. Analyze the generated scenarios in detail using `analyze_uncertainty_scenarios.ipynb`
2. Run the generation for all templates using multiple runs (0, 1, 2) to ensure consistency
3. Compare scenarios across different uncertainty types to identify patterns and insights